In [ ]:
"""
Cash Flow / Balance Monitoring — Structured Streaming
-------------------------------------------------------
Builds on generate_data.ipynb (writes files to ./data) and
account_stream.ipynb (streaming schema/reader).

Key idea: balance = sum(Credits) - sum(Debits) is ORDER-INDEPENDENT.
So we don't need window functions or manual upsert logic — a plain
groupBy + sum in `complete` output mode gives Spark's built-in
stateful streaming aggregation, and it keeps the running total
correct as new files land in ./data.
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, when
from pyspark.sql.types import StructType


def create_spark_session():
    spark = SparkSession.builder.appName("BalanceMonitoring").getOrCreate()
    spark.conf.set("spark.sql.shuffle.partitions", 1)  # single-machine, small data
    return spark


def create_datastreamReader(spark):
    schema = (
        StructType()
        .add("Acc No", "integer")
        .add("Transaction", "string")
        .add("Time Stamp", "string")
        .add("Amount", "integer")
    )

    streaming_df = (
        spark.readStream
        .schema(schema)
        .option("maxFilesPerTrigger", 1)   # one file per trigger, matches generate_data output cadence
        .json("./data")
    )

    accountDF = (
        streaming_df
        .withColumn("Times", to_timestamp(col("Time Stamp")))
        .drop("Time Stamp")
        .withColumn(
            "signed_amount",
            when(col("Transaction") == "C", col("Amount"))
            .when(col("Transaction") == "D", -col("Amount"))
            .otherwise(0)
        )
    )
    return accountDF


def get_live_balances(accountDF):
    """
    Running balance per account, updated as each new file arrives.
    complete mode + groupBy/sum = Spark maintains cumulative state for you.
    """
    balances = (
        accountDF
        .groupBy("Acc No")
        .sum("signed_amount")
        .withColumnRenamed("sum(signed_amount)", "balance")
        .orderBy("Acc No")
    )

    query = (
        balances.writeStream
        .queryName("live_account_balances")
        .format("console")
        .outputMode("complete")
        .option("truncate", "false")
        .trigger(processingTime="5 seconds")
        .start()
    )
    return query


def get_live_balances_to_memory(accountDF):
    """
    Same as above, but writes to an in-memory table you can query
    with spark.sql("SELECT * FROM account_balances ORDER BY balance DESC")
    while the stream is running — handy for a live dashboard/notebook.
    """
    balances = (
        accountDF
        .groupBy("Acc No")
        .sum("signed_amount")
        .withColumnRenamed("sum(signed_amount)", "balance")
    )

    query = (
        balances.writeStream
        .queryName("account_balances")   # becomes the table name for spark.sql
        .format("memory")
        .outputMode("complete")
        .trigger(processingTime="5 seconds")
        .start()
    )
    return query


if __name__ == "__main__":
    spark = create_spark_session()
    accountDF = create_datastreamReader(spark)

    query = get_live_balances(accountDF)
    # query = get_live_balances_to_memory(accountDF)  # use this + spark.sql(...) in a notebook instead

    query.awaitTermination()